In [ ]:
# Import required libraries
import os
import geopandas as gpd
import pandas as pd
import time
import requests
from shapely.geometry import Point, Polygon, MultiPolygon

def down_object(maxy, miny, maxx, minx, tags, crs):
    URL = 'https://api.ohsome.org/v1/elements/geometry'
    bbox = f"{minx:.4f},{miny:.4f},{maxx:.4f},{maxy:.4f}"  # (min_lon, min_lat, max_lon, max_lat)
    payload = {
        "bboxes": bbox,
        "time": "2024-12-01",
        "filter": f"{tags}"
    }
    response = requests.post(URL, data=payload)
    if response.status_code == 200:
        result = response.json()
        locations = []
        for feature in result.get("features", []):
            properties = feature["properties"]
            geom = feature["geometry"]
            name = properties.get("name", "Unknown")
            # Point, Polygon, LineString
            if geom["type"] == "Point":
                geometrys = Point(geom["coordinates"])
            elif geom["type"] == "Polygon":
                geometrys = Polygon(geom["coordinates"][0])
            elif geom["type"] == 'MultiPolygon':
                geometrys = MultiPolygon([Polygon(p[0]) for p in geom["coordinates"]])
            locations.append({"osmid": name, "geometry": geometrys})
    # GeoDataFrame
    objects = gpd.GeoDataFrame(locations, geometry="geometry", crs="EPSG:4326")
    objects = objects.to_crs(f'EPSG:{crs}')
    objects.set_crs={f'EPSG:{crs}'}
    return objects

def overall_process(bound_data, city, crs):
    bound_data = bound_data.to_crs('EPSG:4326')
    bound_data.set_crs={'EPSG:4326'}
    minx, miny, maxx, maxy = bound_data.geometry.total_bounds
    all_retailer = gpd.GeoDataFrame()
    #download retail units
    #amenity tag
    crs = crs
    tag_list = ['restaurant', 'cafe', 'fast_food', 'bank', 'pharmacy', 'bar', 'pub',
                'pub','post_office', 'marketplace', 'nightclub', 'bureau_de_change', 'food_court']
    for tag in tag_list:
        try:
            tags = f'amenity={tag}'
            retailers = down_object(maxy, miny, maxx, minx, tags, crs)
            retailers = retailers.reset_index(drop = False)
            retailers = retailers[['osmid', 'geometry']]
            all_retailer = pd.concat([all_retailer, retailers])
        except Exception as e:
            print(f"Failed to download tag {tag}: {e}")
            pass
    #shop_tag
    try:
        tags = 'shop=*'
        all_shops = down_object(maxy, miny, maxx, minx, tags, crs)
        all_shops = all_shops.reset_index(drop = False)
        all_shops = all_shops[['osmid', 'geometry']]
        all_retailer = pd.concat([all_retailer, all_shops])
    except Exception as e:
        print(f"Failed to download shops: {e}")
        pass

    # Remove duplicates and save all retailer data to parquet
    all_retailer = all_retailer.drop_duplicates(subset=['osmid'])
    retailer_dir = './retailers/' + city + '_retailers.parquet'
    if not all_retailer.empty:
        all_retailer.to_parquet(retailer_dir)

    # Download retail buildings (e.g., retail stores and supermarkets)
    building_list = ['retail', 'supermarket']
    retail = gpd.GeoDataFrame()

    for building in building_list:
        try:
            tags = f'building={building}'
            temp_retail = down_object(maxy, miny, maxx, minx, tags, crs)
            temp_retail = temp_retail.reset_index(drop = False)[['osmid', 'geometry']]
            retail = pd.concat([retail, temp_retail])
        except Exception as e:
            print(f"Failed to download {building} retail buildings: {e}")
            pass
    
    # Remove duplicates and save retail buildings data to parquet
    retail = retail.drop_duplicates(subset=['osmid'])
    retail_dir = './retail/' + city + '_retail.parquet'
    if not retail.empty:
        retail.to_parquet(retail_dir)

    # Download office buildings and office-related points
    office = gpd.GeoDataFrame()
    try:
        tags = 'building=office'
        temp_office = down_object(maxy, miny, maxx, minx, tags, crs)
        temp_office = temp_office.reset_index(drop = False)[['osmid', 'geometry']]
        retail = pd.concat([office, temp_office])
    except Exception as e:
        print(f"Failed to download office buildings: {e}")
        pass
    
    # Download additional office-related features using the 'office' tag
    tag_list = [
        'accountant', 'government', 'company', 'construction_company', 'consulting', 'coworking', 'diplomatic', 
        'employment_agency', 'financial', 'financial_advisor', 'government', 'it', 'lawyer', 'newspaper', 
        'ngo', 'tax_advisor'
    ]
    for tag in tag_list:
        try:
            tags = f'office={tag}'
            temp_office = down_object(maxy, miny, maxx, minx, tags, crs)
            temp_office = temp_office.reset_index(drop=False)[['osmid', 'geometry']]
            office = pd.concat([office, temp_office])
        except Exception as e:
            print(f"Failed to download additional office features: {e}")
            pass

    # Remove duplicates and save office data to parquet
    office = office.drop_duplicates(subset=['osmid'])
    office_dir = './office/' + city + '_office.parquet'
    if not office.empty:
        office.to_parquet(office_dir)

In [ ]:
# Set working directory and load metropolitan area boundaries
os.chdir("/Users/jpg23/data/downtownrecovery/commercial_districts/commercial_districts_paper/")
ma_america = gpd.read_parquet("./top300metros.parquet")

In [ ]:
# sorted(ma_america['name'].unique())

In [ ]:
# Filter to only metro regions that are not already in Byeonghwa's dataset
existing = gpd.read_file('/Users/jpg23/data/downtownrecovery/commercial_districts/byeonghwa_commercial_districts.geojson')
existing['name'] = existing['MSA_NAME'] + ', ' + existing['STATE/PROVINCE']

In [ ]:
existing_names = set(existing['name'])

In [ ]:
# type(existing_names)

In [ ]:
# len(existing_names)

In [ ]:
# existing_names

In [ ]:
# there should be 4 - but if there are fewer (i.e., I download too much) that's ok also - we can filter down
# len_before = len(existing_names)
existing_names.update(['Ottawa - Gatineau, ON-QC', 'Louisville, KY-IN'])
# len_after = len(existing_names)
# print(f"existing_names length went from {len_before} to {len_after}")

In [ ]:
# filter data to only places that aren't already in Byeonghwa's dataset
ma_new = ma_america[~ma_america['name'].isin(existing_names)]
# ma_new.shape[0] # should be 300-70 = 230 (if it's a bit longer that's ok for now)

In [ ]:
ma_new['name'].unique()

In [ ]:
# import re
# unique_ma_new = ma_new['name'].unique()
# first_words_ma_new = [re.split(r'[- ,/]', item)[0] for item in unique_ma_new]
# # first_words_ma_new

In [ ]:
# first_words_existing = [re.split(r'[- ,/]', item)[0] for item in existing_names]
# # first_words_existing

In [ ]:
# common = list(set(first_words_ma_new) & set(first_words_existing))
# common

In [ ]:
# # 4 of these will additionally need to be filtered out (whichever are already in the below list from 'existing_names')
# [s for s in unique_ma_new if any(s.startswith(f) for f in common)]

In [ ]:
# [s for s in existing_names if any(s.startswith(f) for f in common)] # which of these match the list above?

In [ ]:
# it already ran for these places, so skip them:
already_done = ['Grand Rapids-Kentwood, MI', 'Rochester, NY', 'Worcester, MA-CT',
                'Bridgeport-Stamford-Norwalk, CT', 'Greenville-Anderson, SC',
                'Albany-Schenectady-Troy, NY', 'Knoxville, TN',
                'McAllen-Edinburg-Mission, TX', 'Baton Rouge, LA', 'New Haven-Milford, CT',
                'Allentown-Bethlehem-Easton, PA-NJ', 'North Port-Sarasota-Bradenton, FL',
                'Oxnard-Thousand Oaks-Ventura, CA', 'Quebec, QC', 'Columbia, SC',
                'Dayton-Kettering, OH', 'Charleston-North Charleston, SC', 'Hamilton, ON',
                'Stockton, CA', 'Greensboro-High Point, NC', 'Cape Coral-Fort Myers, FL',
                'Boise City, ID', 'Little Rock-North Little Rock-Conway, AR',
                'Lakeland-Winter Haven, FL', 'Des Moines-West Des Moines, IA',
                'Akron, OH', 'Poughkeepsie-Newburgh-Middletown, NY',
                'Ogden-Clearfield, UT', 'Springfield, MA', 'Madison, WI',
                'Winston-Salem, NC', 'Provo-Orem, UT',
                'Deltona-Daytona Beach-Ormond Beach, FL', 'Syracuse, NY',
                'Durham-Chapel Hill, NC', 'Toledo, OH',
                'Augusta-Richmond County, GA-SC',
                'Palm Bay-Melbourne-Titusville, FL', 'Harrisburg-Carlisle, PA',
                'Jackson, MS', 'Spokane-Spokane Valley, WA',
                'Kitchener - Cambridge - Waterloo, ON',
                'Scranton--Wilkes-Barre, PA', 'Chattanooga, TN-GA',
                'Lancaster, PA', 'Portland-South Portland, ME', 'Modesto, CA',
                'Fayetteville-Springdale-Rogers, AR',
                'Youngstown-Warren-Boardman, OH-PA', 'Lansing-East Lansing, MI',
                'Fayetteville, NC', 'Lexington-Fayette, KY',
                'Pensacola-Ferry Pass-Brent, FL',
                'Myrtle Beach-Conway-North Myrtle Beach, SC-NC',
                'Port St. Lucie, FL', 'Huntsville, AL', 'Reno, NV',
                'Santa Rosa-Petaluma, CA', 'Lafayette, LA', 'Killeen-Temple, TX',
                'Springfield, MO', 'Visalia, CA', 'Asheville, NC', 'Halifax, NS',
                'York-Hanover, PA', 'Vallejo, CA', 'Santa Maria-Santa Barbara, CA',
                'Salinas, CA', 'St. Catharines - Niagara, ON', 'Salem, OR',
                'Mobile, AL', 'Reading, PA', 'Manchester-Nashua, NH',
                'Windsor, ON', 'Corpus Christi, TX', 'Salisbury, MD-DE',
                'Brownsville-Harlingen, TX', 'Fort Wayne, IN',
                'Gulfport-Biloxi, MS', 'Oshawa, ON', 'Savannah, GA', 'Flint, MI',
                'Peoria, IL', 'Canton-Massillon, OH', 'Anchorage, AK',
                'Victoria, BC', 'Beaumont-Port Arthur, TX',
                'Shreveport-Bossier City, LA', 'Tallahassee, FL', 'Montgomery, AL',
                'Trenton-Princeton, NJ', 'Davenport-Moline-Rock Island, IA-IL',
                'Eugene-Springfield, OR', 'Naples-Marco Island, FL', 'Ocala, FL',
                'Ann Arbor, MI', 'Hickory-Lenoir-Morganton, NC',
                'Fort Collins, CO', 'Huntington-Ashland, WV-KY-OH',
                'Gainesville, FL', 'Lincoln, NE', 'Rockford, IL', 'Greeley, CO',
                'Spartanburg, SC', 'Boulder, CO', 'Green Bay, WI',
                'Columbus, GA-AL', 'South Bend-Mishawaka, IN-MI',
                'Clarksville, TN-KY', 'Lubbock, TX', 'Saskatoon, SK',
                'Roanoke, VA', 'Evansville, IN-KY', 'Kingsport-Bristol, TN-VA',
                'Kennewick-Richland, WA', 'Hagerstown-Martinsburg, MD-WV',
                'Olympia-Lacey-Tumwater, WA', 'Duluth, MN-WI', 'Utica-Rome, NY',
                'Wilmington, NC', 'Crestview-Fort Walton Beach-Destin, FL',
                'Longview, TX', 'Merced, CA', 'San Luis Obispo-Paso Robles, CA',
                'Waco, TX', 'Sioux Falls, SD', 'Cedar Rapids, IA',
                'Bremerton-Silverdale-Port Orchard, WA',
                'Atlantic City-Hammonton, NJ', 'Tuscaloosa, AL', 'Erie, PA',
                'College Station-Bryan, TX', 'Amarillo, TX',
                'Santa Cruz-Watsonville, CA', 'Norwich-New London, CT',
                'Laredo, TX', 'Lynchburg, VA', 'Kalamazoo-Portage, MI',
                'Charleston, WV', 'Yakima, WA', 'Fargo, ND-MN', 'Regina, SK',
                'Binghamton, NY', 'Fort Smith, AR-OK', 'Appleton, WI',
                'Prescott Valley-Prescott, AZ', 'Tyler, TX',
                'Daphne-Fairhope-Foley, AL', 'Macon-Bibb County, GA', 'Topeka, KS',
                'Barnstable Town, MA', 'Sherbrooke, QC', 'Bellingham, WA',
                'Rochester, MN', 'Burlington-South Burlington, VT',
                'Lafayette-West Lafayette, IN', 'Champaign-Urbana, IL',
                'Medford, OR', 'Kelowna, BC', 'Charlottesville, VA',
                'Lebanon, NH-VT Micro Area', 'Las Cruces, NM',
                'Hilton Head Island-Bluffton, SC', 'Lake Charles, LA',
                'Athens-Clarke County, GA', 'Lake Havasu City-Kingman, AZ',
                'Chico, CA', 'Barrie, ON', "St. John's, NL", 'Columbia, MO',
                'Springfield, IL', 'Johnson City, TN', 'Elkhart-Goshen, IN',
                'Houma-Thibodaux, LA', 'Monroe, LA', 'Gainesville, GA', 'Yuma, AZ',
                'Jacksonville, NC', 'Hilo, HI Micro Area', 'Florence, SC',
                'St. Cloud, MN', 'Bend, OR', 'Racine, WI']

ma_sub = ma_new[~ma_new['name'].isin(already_done)]
print(f"Number of places before: {len(ma_new)}; number of places after: {len(ma_sub)}")

In [ ]:
# Loop over each metro area
for x in ma_sub.index:
    temp_ma = ma_sub.loc[ma_sub.index == x].copy()
    city = temp_ma['name'].tolist()[0].split(',')[0].replace(' ', '-')
    print(f"\n{city}\n*************************************")
    start_time = time.time()
    overall_process(temp_ma, city, crs=4326)
    end_time = time.time()
    elapsed = (end_time - start_time) / 60
    print(f"overall_process() run for {city} - it took {elapsed:.2f} minute(s)")